In [ ]:
import cv2
import time
import threading
cap = cv2.VideoCapture()

## YOLO

In [ ]:
import ultralytics
ultralytics.checks()

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data="/home/hospital/yolo/pinkycare.yolov8/finish.yaml",
    epochs=100,
    imgsz=640,
    batch=16,      # GPU 메모리 부족하면 8
    device=0,      # 로컬 GPU
    name="delivery_train",
)

In [ ]:
# val 이미지들로 predict 돌려보기

from ultralytics import YOLO

model = YOLO("/home/hospital/yolo/runs/detect/delivery_train/weights/best.pt")

results = model.predict(
    "/home/hospital/yolo/pinkycare.yolov8/yolo_images/images/val",  # val 폴더 통째로
    save=True,          # 결과 이미지 저장
    imgsz=640,
    conf=0.5,
    save_conf=True,
    name="val_predict",
)

## Webcam

In [ ]:
import cv2
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
cap = cv2.VideoCapture(1)
print("캡쳐 시작")
while True:
    ret, frame = cap.read()
    if not ret:
        break
    results = model(frame, stream=True, device='cuda', conf=0.5)

    for r in results:
        annotated_frame = r.plot()

    cv2.imshow("YOLO Real-time", annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2, threading, time
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

CLASS_NAME = "실패"
N_SHOTS    = 100
CAM_INDEX  = 4

save_dir = Path(f"yolo1/{CLASS_NAME}")
save_dir.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(CAM_INDEX)
if not cap.isOpened():
    raise RuntimeError(f"웹캠 인덱스 {CAM_INDEX} 못 엶")

img   = widgets.Image(format='jpeg', width=480)
btn   = widgets.Button(description="촬영 (c)", button_style='success')
stop  = widgets.Button(description="종료 (q)", button_style='danger')
label = widgets.Label(value="0장 저장됨")
display(widgets.VBox([img, widgets.HBox([btn, stop]), label]))

state = {"count": 0, "run": True, "frame": None}

def loop():
    while state["run"]:
        ok, frame = cap.read()
        if not ok:
            continue
        state["frame"] = frame
        _, buf = cv2.imencode(".jpg", frame)
        img.value = buf.tobytes()
        time.sleep(0.05)
    cap.release()

def shoot(_):
    f = state["frame"]
    if f is None:
        return
    path = save_dir / f"{CLASS_NAME}_{state['count']:03d}.jpg"
    cv2.imwrite(str(path), f)
    state["count"] += 1
    label.value = f"{state['count']}장 저장됨  (마지막: {path.name})"
    if state["count"] >= N_SHOTS:
        state["run"] = False
        label.value = f"완료! 총 {state['count']}장"

def end(_):
    state["run"] = False
    label.value = f"종료됨. 총 {state['count']}장"

btn.on_click(shoot)
stop.on_click(end)
threading.Thread(target=loop, daemon=True).start()

In [ ]:
cat ~/yolo/pinkycare.yolov8/data.yaml
echo "-----"
cat ~/yolo/pinkycare.yolov8/finish.yaml